In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
df_nq = yf.download("NQ=F", interval="1m", period="7d")

In [ ]:
df_nq.index = df_nq.index.tz_convert("America/New_York")
df_nq["date"] = df_nq.index.date
df_nq.columns = df_nq.columns.get_level_values(0)

In [ ]:
rth = df_nq.between_time("09:30", "15:59").copy()
rth["date"] = rth.index.date
one_day = rth[rth["date"] == rth["date"].iloc[0]]
opening_window = one_day.between_time("09:30", "09:59")

In [ ]:
atr_value = 300
threshold_fraction = 0.12

In [ ]:
session_open = one_day.iloc[0]["Open"]

trigger_distance = atr_value * threshold_fraction

bullish_trigger = session_open + trigger_distance
bearish_trigger = session_open - trigger_distance


In [ ]:
event_found = False
event_direction = None
entry_price = None
entry_time = None

for timestamp, row in opening_window.iterrows():
    bullish_touched = row["High"] >= bullish_trigger
    bearish_touched = row["Low"] <= bearish_trigger

    # Both thresholds touched in the same candle
    if bullish_touched and bearish_touched:
        event_found = True
        event_direction = "conflict_loss"
        entry_time = timestamp
        break

    # Bullish trigger
    elif bullish_touched:
        event_found = True
        event_direction = "bullish"
        entry_price = bullish_trigger
        entry_time = timestamp
        break

    # bearish trigger
    elif bearish_touched:
        event_found = True
        event_direction = "bearish"
        entry_price = bearish_trigger
        entry_time = timestamp
        break

In [ ]:
print("Event found: ", event_found)
print("Direction: ", event_direction)
print("Entry Price: ", entry_price)
print("Entry time: ", entry_time)

In [ ]:
# def detect_opening_impulse(day_df, atr_value, threshold_fraction):

#     session_open = day_df.iloc[0][["open"]]

#     trigger_distance = atr_value * threshold_fraction

#     bullish_trigger = session_open + trigger_distance
#     bearish_trigger = session_open - trigger_distance

In [ ]:
target_multiple = 1.0
stop_multiple = 1.0

In [ ]:
if event_direction in ["bullish", "bearish"]:
    target_distance = trigger_distance * target_multiple
    stop_distance = trigger_distance * stop_multiple

    if event_direction == "bullish":
        target_price = entry_price + target_distance
        stop_price = entry_price - stop_distance

    elif event_direction == "bearish":
        target_price = entry_price - target_distance
        stop_price = entry_price + stop_distance

In [ ]:
print("Entry price: ", entry_price)
print("Target price: ", target_price)
print("Stop price: ", stop_price)

In [ ]:
post_entry = one_day.loc[entry_time:]
trade_result = None

In [ ]:
for timestamp, row in post_entry.iterrows():

    if event_direction == "bullish":
        target_hit = row["High"] >= target_price
        stop_hit = row["Low"] <= stop_price

    elif event_direction == "bearish":
        target_hit = row["Low"] <= target_price
        stop_hit = row["High"] >= stop_price

    if target_hit and stop_hit:
        trade_result = "loss"
        break

    elif stop_hit:
        trade_result = "loss"
        break

    elif target_hit:
        trade_result = "win"
        break

if trade_result is None:
    trade_result = "unresolved"

In [ ]:
print("Trade result:", trade_result)